# 실적 발표 이후 PER의 시차 변화 — AAPL 파일럿

**연구 질문:** 분기 실적 발표(rdq)로 새로운 EPS가 공개된 뒤, 다음 발표까지 약 90일 동안
TTM PER은 어떤 경로로 움직이는가?

**동기:** PER의 분모(E)는 발표일에만 점프하고 분자(P)는 매일 움직인다. 따라서 발표 직후부터의
PER 경로는 "시장이 새 실적 정보를 가격에 얼마나 빨리, 어떤 방향으로 반영하는가"를 보여주는
렌즈가 된다 (post-earnings drift 문헌의 PER 버전).

**데이터 (WRDS):**
- `comp.fundq` — 분기 EPS(epsfxq), 실적 발표일(rdq), 주식조정계수(ajexq)
- `crsp.dsf` — 일별 주가(prc), 가격조정계수(cfacpr)
- `crsp.ccmxpf_linktable` — gvkey↔permno 연결. **ticker 조인은 재사용 문제로 쓰지 않는다.**

**설계:** AAPL(gvkey 001690) 단일 종목 파일럿. 각 발표 이벤트를 day 0으로 정렬(event-time)해
발표 후 0~90일의 TTM PER 경로를 day-0 대비 지수로 정규화하고, 이벤트 전체의 중앙값·IQR을 본다.

> **실행에는 WRDS 학술 계정이 필요하다** (`WRDS_USERNAME` 환경변수 설정).

In [ ]:
import os
import wrds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

db = wrds.Connection(wrds_username=os.environ["WRDS_USERNAME"])
print("connected")

## 1. 펀더멘털 + 링크테이블

CCM 링크테이블에서 primary link(LU/LC × P/C)만 취하고, 회계일(datadate)이 링크 유효기간에
들어오는 행만 조인한다.

In [ ]:
fund = db.raw_sql("""
    WITH link AS (
        SELECT gvkey, lpermno AS permno,
               linkdt, COALESCE(linkenddt, CURRENT_DATE) AS linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE linktype IN ('LU', 'LC')
          AND linkprim IN ('P', 'C')
          AND gvkey = '001690'          -- AAPL
    )
    SELECT f.gvkey, l.permno, f.datadate, f.rdq, f.epsfxq, f.ajexq
    FROM comp.fundq f
    JOIN link l
      ON f.gvkey = l.gvkey
     AND f.datadate BETWEEN l.linkdt AND l.linkenddt
    WHERE f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.consol  = 'C'
      AND f.popsrc  = 'D'
      AND f.rdq IS NOT NULL
      AND f.datadate >= '2010-01-01'
    ORDER BY f.datadate
""", date_cols=["datadate", "rdq"])
print(fund.shape)
fund.tail()

## 2. 일별 수정주가

`prc`의 음수(호가 중값 표시)는 절댓값으로, 분할은 `cfacpr`로 조정한다.

In [ ]:
permno = int(fund["permno"].iloc[0])
px = db.raw_sql("""
    SELECT date, ABS(prc) / NULLIF(cfacpr, 0) AS adj_prc
    FROM crsp.dsf
    WHERE permno = %(permno)s
      AND date >= '2010-06-01'
    ORDER BY date
""", params={"permno": permno}, date_cols=["date"])
print(px.shape)
px.tail()

## 3. Point-in-time TTM EPS → 일별 TTM PER

- 분할 조정 EPS = `epsfxq / ajexq` (CRSP 가격 조정과 같은 기준으로 맞춤)
- TTM EPS = 최근 4개 분기 합 — **발표일(rdq)부터** 유효한 것으로 취급 (look-ahead 방지)
- `merge_asof(backward)`로 각 거래일에 "그 시점까지 발표된" 최신 TTM EPS를 붙인다

In [ ]:
fund = fund.sort_values("datadate").reset_index(drop=True)
fund["eps_adj"] = fund["epsfxq"] / fund["ajexq"]
fund["ttm_eps"] = fund["eps_adj"].rolling(4).sum()

events = (fund.dropna(subset=["ttm_eps"])
              .sort_values("rdq")[["rdq", "ttm_eps"]]
              .drop_duplicates(subset="rdq", keep="last"))

daily = pd.merge_asof(px.sort_values("date"), events,
                      left_on="date", right_on="rdq", direction="backward")
daily = daily.dropna(subset=["ttm_eps"])
daily = daily[daily["ttm_eps"] > 0]
daily["ttm_per"] = daily["adj_prc"] / daily["ttm_eps"]
daily["days_since"] = (daily["date"] - daily["rdq"]).dt.days
daily.tail()

## 4. 이벤트-타임 정렬: 발표 후 0~90일 PER 경로

각 발표 이벤트의 day-0 PER을 100으로 정규화해 이벤트 간 비교 가능하게 만든다.

In [ ]:
paths = []
for rdq, g in daily.groupby("rdq"):
    g = g[g["days_since"] <= 90].sort_values("days_since")
    if len(g) < 30:
        continue
    base = g["ttm_per"].iloc[0]
    paths.append(pd.DataFrame({"days_since": g["days_since"],
                               "per_index": g["ttm_per"] / base * 100,
                               "rdq": rdq}))
panel = pd.concat(paths, ignore_index=True)

stats = (panel.groupby("days_since")["per_index"]
              .agg(median="median",
                   q25=lambda s: s.quantile(0.25),
                   q75=lambda s: s.quantile(0.75)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stats.index, stats["median"], color="#1f77b4", linewidth=2,
        label=f"median ({panel['rdq'].nunique()} events)")
ax.fill_between(stats.index, stats["q25"], stats["q75"], alpha=0.2, label="IQR")
ax.axhline(100, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("days since earnings announcement")
ax.set_ylabel("TTM PER (day 0 = 100)")
ax.set_title("AAPL: TTM PER drift after earnings announcements")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("aapl_per_drift.png", dpi=120)
plt.show()

## 해석 가이드 및 한계

**보는 법:**
- 중앙값 경로가 100 위로 표류하면 발표 후 주가가 새 EPS 대비 계속 리레이팅된다는 뜻
  (post-earnings drift와 일관), 100 근처 횡보면 발표 시점에 정보가 즉시 반영된다는 뜻이다.
- IQR 폭은 이벤트 간 이질성 — 서프라이즈 방향·크기에 따라 경로가 갈리는 정도를 보여준다.

**한계와 확장 방향:**
1. 단일 종목(AAPL) 파일럿이므로 일반화 불가 — S&P500 크로스섹션으로 확장해
   어닝 서프라이즈(SUE) 부호별로 경로를 나눠 보는 것이 다음 단계.
2. TTM EPS가 음수→양수로 전환되는 구간은 PER이 정의되지 않아 제외했다.
3. `ajexq`(Compustat)와 `cfacpr`(CRSP)의 조정 기준일 차이로 미세한 불일치가 있을 수 있다.